In [17]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 transformers torch sentencepiece accelerate pandas datasets "bitsandbytes>=0.40.0" -q

In [18]:
%%writefile db.py
import sqlite3
import bcrypt
import datetime
import time
import os

# Google Drive DB support
GDRIVE_DIR = "/content/drive/MyDrive/TextMorph"

import sys
if "google.colab" in sys.modules:
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            drive.mount('/content/drive')
    except Exception as e:
        print(f"⚠️ Could not mount drive: {e}")

if os.path.exists("/content/drive/MyDrive"):
    if not os.path.exists(GDRIVE_DIR):
        os.makedirs(GDRIVE_DIR)
    DB_NAME = os.path.join(GDRIVE_DIR, "users.db")
else:
    DB_NAME = "users.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
        # Users table
    c.execute('''CREATE TABLE IF NOT EXISTS users (
        email TEXT PRIMARY KEY,
        password BLOB,
        avatar TEXT,
        created_at TEXT
    )''')

    # Add missing columns
    try:
        c.execute("ALTER TABLE users ADD COLUMN role TEXT DEFAULT 'user'")
    except sqlite3.OperationalError:
        pass

    try:
        c.execute("ALTER TABLE users ADD COLUMN locked INTEGER DEFAULT 0")
    except sqlite3.OperationalError:
        pass

    # Password History table
    c.execute('''CREATE TABLE IF NOT EXISTS password_history
                 (id INTEGER PRIMARY KEY AUTOINCREMENT,
                  email TEXT,
                  password BLOB,
                  set_at TEXT,
                  FOREIGN KEY(email) REFERENCES users(email))''')

    # Login Attempts table (Rate Limiting)
    c.execute('''CREATE TABLE IF NOT EXISTS login_attempts
                 (email TEXT PRIMARY KEY,
                  attempts INTEGER DEFAULT 0,
                  last_attempt REAL)''')

    # Feedback table
    c.execute('''CREATE TABLE IF NOT EXISTS feedback
                 (id INTEGER PRIMARY KEY AUTOINCREMENT,
                  email TEXT,
                  original_text TEXT,
                  generated_text TEXT,
                  task_type TEXT,
                  rating INTEGER,
                  comments TEXT,
                  created_at TEXT)''')

    # Activity History table
    c.execute('''CREATE TABLE IF NOT EXISTS activity_history
                 (id INTEGER PRIMARY KEY AUTOINCREMENT,
                  email TEXT,
                  activity_type TEXT,
                  details TEXT,
                  model_used TEXT,
                  created_at TEXT)''')
    c.execute('''CREATE TABLE IF NOT EXISTS active_users
             (email TEXT PRIMARY KEY,
              login_time TEXT,
              last_activity TEXT)''')

# System Logs table
    c.execute('''CREATE TABLE IF NOT EXISTS system_logs
             (id INTEGER PRIMARY KEY AUTOINCREMENT,
              log_type TEXT,
              message TEXT,
              created_at TEXT)''')

# Feature Usage table
# Feature Usage table
    c.execute('''CREATE TABLE IF NOT EXISTS feature_usage
             (id INTEGER PRIMARY KEY AUTOINCREMENT,
              email TEXT,
              feature_name TEXT,
              used_at TEXT)''')
    c.execute('''CREATE TABLE IF NOT EXISTS usage_analytics
             (id INTEGER PRIMARY KEY AUTOINCREMENT,
              email TEXT,
              feature TEXT,
              model TEXT,
              language TEXT,
              created_at TEXT)''')

    conn.commit()
    conn.close()

def _get_timestamp():
    return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

def register_user(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    try:
        salt = bcrypt.gensalt()
        hashed = bcrypt.hashpw(password.encode('utf-8'), salt)
        now = _get_timestamp()
        c.execute("INSERT INTO users (email, password, created_at) VALUES (?, ?, ?)", (email, hashed, now))
        c.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

def authenticate_user(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    if data:
        stored_hash = data[0]
        if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
            _reset_attempts(email)
            return True
    _record_failed_attempt(email)
    return False
    c.execute("SELECT password, locked FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    if data:
      stored_hash, locked = data
      if locked:
        return False

def check_is_old_password(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password, set_at FROM password_history WHERE email = ? ORDER BY set_at DESC", (email,))
    history = c.fetchall()
    conn.close()
    for stored_hash, set_at in history:
        if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
            return set_at
    return None

def check_password_reused(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM password_history WHERE email = ?", (email,))
    history = c.fetchall()
    conn.close()
    for (stored_hash,) in history:
        if bcrypt.checkpw(new_password.encode('utf-8'), stored_hash):
            return True
    return False

def check_user_exists(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT 1 FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    return data is not None

def update_password(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(new_password.encode('utf-8'), salt)
    now = _get_timestamp()
    c.execute("UPDATE users SET password = ? WHERE email = ?", (hashed, email))
    c.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now))
    conn.commit()
    conn.close()

# --- Rate Limiting ---
MAX_ATTEMPTS = 3
LOCKOUT_SECONDS = 60

def _record_failed_attempt(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    now = time.time()
    c.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email = ?", (email,))
    row = c.fetchone()
    if row:
        attempts, last = row
        if now - last > LOCKOUT_SECONDS:
            c.execute("UPDATE login_attempts SET attempts = 1, last_attempt = ? WHERE email = ?", (now, email))
        else:
            c.execute("UPDATE login_attempts SET attempts = ?, last_attempt = ? WHERE email = ?", (attempts + 1, now, email))
    else:
        c.execute("INSERT INTO login_attempts (email, attempts, last_attempt) VALUES (?, 1, ?)", (email, now))
    conn.commit()
    conn.close()

def _reset_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
    conn.commit()
    conn.close()
# --- Active User Tracking ---

def set_user_active(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    now = _get_timestamp()

    c.execute("""
        INSERT INTO active_users (email, login_time, last_activity)
        VALUES (?, ?, ?)
        ON CONFLICT(email) DO UPDATE SET last_activity=excluded.last_activity
    """, (email, now, now))

    conn.commit()
    conn.close()


def update_user_activity(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    now = _get_timestamp()

    c.execute("UPDATE active_users SET last_activity=? WHERE email=?", (now, email))

    conn.commit()
    conn.close()


def remove_active_user(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("DELETE FROM active_users WHERE email=?", (email,))

    conn.commit()
    conn.close()


def get_active_users():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("SELECT email, login_time, last_activity FROM active_users")

    users = c.fetchall()
    conn.close()
    return users
# --- System Logs ---

def log_system_event(log_type, message):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    now = _get_timestamp()

    c.execute("""
        INSERT INTO system_logs (log_type, message, created_at)
        VALUES (?, ?, ?)
    """, (log_type, message, now))

    conn.commit()
    conn.close()


def get_system_logs():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
        SELECT log_type, message, created_at
        FROM system_logs
        ORDER BY created_at DESC
    """)

    logs = c.fetchall()
    conn.close()
    return logs

def is_rate_limited(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email = ?", (email,))
    row = c.fetchone()
    conn.close()
    if row:
        attempts, last = row
        elapsed = time.time() - last
        if attempts >= MAX_ATTEMPTS and elapsed < LOCKOUT_SECONDS:
            return True, LOCKOUT_SECONDS - elapsed
    return False, 0
# --- Feature Usage Tracking ---

def track_feature_usage(email, feature_name):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    now = _get_timestamp()

    c.execute("""
        INSERT INTO feature_usage (email, feature_name, used_at)
        VALUES (?, ?, ?)
    """, (email, feature_name, now))

    conn.commit()
    conn.close()


def get_feature_usage_stats():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
        SELECT feature_name, COUNT(*)
        FROM feature_usage
        GROUP BY feature_name
        ORDER BY COUNT(*) DESC
    """)

    stats = c.fetchall()
    conn.close()
    return stats

# --- Feedback System ---
def save_feedback(email, original_text, generated_text, task_type, rating, comments):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    now = _get_timestamp()
    c.execute("INSERT INTO feedback (email, original_text, generated_text, task_type, rating, comments, created_at) VALUES (?, ?, ?, ?, ?, ?, ?)",
              (email, original_text[:500], generated_text[:500], task_type, rating, comments, now))
    conn.commit()
    conn.close()

def get_all_feedback():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT id, email, task_type, rating, comments, created_at FROM feedback ORDER BY created_at DESC")
    feedback = c.fetchall()
    conn.close()
    return feedback

# --- Activity History System ---
def log_activity(email, activity_type, details, model_used):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    now = _get_timestamp()
    c.execute("INSERT INTO activity_history (email, activity_type, details, model_used, created_at) VALUES (?, ?, ?, ?, ?)",
              (email, activity_type, details, model_used, now))
    conn.commit()
    conn.close()

def get_user_activity(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT activity_type, details, model_used, created_at FROM activity_history WHERE email = ? ORDER BY created_at DESC", (email,))
    activities = c.fetchall()
    conn.close()
    return activities

# --- Admin Functions ---
def get_all_users():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT email, role, locked, created_at FROM users ORDER BY created_at DESC")
    users = c.fetchall()
    conn.close()
    return users

def delete_user(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM password_history WHERE email = ?", (email,))
    c.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
    c.execute("DELETE FROM feedback WHERE email = ?", (email,))
    c.execute("DELETE FROM users WHERE email = ?", (email,))
    conn.commit()
    conn.close()
def get_user_profile(email):

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("SELECT email, avatar, created_at FROM users WHERE email=?", (email,))
    user = c.fetchone()

    conn.close()

    return user


def update_email(old_email, new_email):

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("UPDATE users SET email=? WHERE email=?", (new_email, old_email))

    conn.commit()
    conn.close()
def check_same_password(email, new_password):
    conn = sqlite3.connect("database.db")
    cursor = conn.cursor()

    cursor.execute("SELECT password FROM users WHERE email=?", (email,))
    result = cursor.fetchone()

    conn.close()

    if result:
        stored_password = result[0]
        return bcrypt.checkpw(new_password.encode(), stored_password)

    return False


def update_avatar(email, avatar_path):

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("UPDATE users SET avatar=? WHERE email=?", (avatar_path, email))

    conn.commit()
    conn.close()
def promote_to_admin(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("UPDATE users SET role='admin' WHERE email=?", (email,))
    conn.commit()
    conn.close()


def lock_user(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("UPDATE users SET locked=1 WHERE email=?", (email,))
    conn.commit()
    conn.close()


def unlock_user(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("UPDATE users SET locked=0 WHERE email=?", (email,))
    conn.commit()
    conn.close()
def log_usage(email, feature, model, language):

    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    now = datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

    c.execute("""
        INSERT INTO usage_analytics (email, feature, model, language, created_at)
        VALUES (?, ?, ?, ?, ?)
    """, (email, feature, model, language, now))

    conn.commit()
    conn.close()
def get_feature_usage():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
        SELECT feature, COUNT(*)
        FROM usage_analytics
        GROUP BY feature
    """)

    data = c.fetchall()
    conn.close()
    return data


def get_model_usage():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
        SELECT model, COUNT(*)
        FROM usage_analytics
        GROUP BY model
    """)

    data = c.fetchall()
    conn.close()
    return data


def get_language_usage():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    c.execute("""
        SELECT language, COUNT(*)
        FROM usage_analytics
        GROUP BY language
    """)

    data = c.fetchall()
    conn.close()
    return data


Overwriting db.py


In [19]:
%%writefile readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text
        self.num_sentences = textstat.sentence_count(text)
        self.num_words = textstat.lexicon_count(text, removepunct=True)
        self.num_syllables = textstat.syllable_count(text)
        self.complex_words = textstat.difficult_words(text)
        self.char_count = textstat.char_count(text)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text),
            "Coleman-Liau": textstat.coleman_liau_index(self.text)
        }


Overwriting readability.py


In [20]:
%%writefile engine.py
import os
import torch
import streamlit as st
import nltk
from nltk.tokenize import sent_tokenize

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

TRANSFORMERS_AVAILABLE = False
BNB_AVAILABLE = False
try:
    from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
    TRANSFORMERS_AVAILABLE = True
    try:
        from transformers import BitsAndBytesConfig
        BNB_AVAILABLE = True
    except ImportError:
        pass
except ImportError as e:
    TRANSFORMERS_AVAILABLE = False

# ─── NLLB Translation Support ───
LANG_CODES = {
    "English": "eng_Latn", "Hindi": "hin_Deva", "Tamil": "tam_Taml",
    "Kannada": "kan_Knda", "Telugu": "tel_Telu", "Marathi": "mar_Deva",
    "Bengali": "ben_Beng"
}

@st.cache_resource(show_spinner=False)
def load_translation_model():
    """Load NLLB-200 distilled model for multilanguage translation"""
    if not TRANSFORMERS_AVAILABLE:
        return None, None
    try:
        model_id = "facebook/nllb-200-distilled-600M"
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_id, device_map="auto")
        return tokenizer, model
    except Exception as e:
        print(f"Warning: Could not load NLLB translation model: {e}")
        return None, None

def translate_text(text, source_lang="English", target_lang="English"):
    """Translate text between supported languages using NLLB-200"""
    if source_lang == target_lang:
        return text

    trans_tok, trans_model = load_translation_model()
    if trans_tok is None or trans_model is None:
        return text  # fallback: return untranslated

    src_code = LANG_CODES.get(source_lang, "eng_Latn")
    tgt_code = LANG_CODES.get(target_lang, "eng_Latn")

    try:
        # Process in chunks to handle long texts
        sentences = sent_tokenize(text)
        chunks = []
        curr_chunk = []
        curr_len = 0
        for s in sentences:
            s_len = len(s.split())
            if curr_len + s_len > 200 and curr_chunk:
                chunks.append(" ".join(curr_chunk))
                curr_chunk = [s]
                curr_len = s_len
            else:
                curr_chunk.append(s)
                curr_len += s_len
        if curr_chunk:
            chunks.append(" ".join(curr_chunk))

        translated_parts = []
        for chunk in chunks:
            trans_tok.src_lang = src_code
            inputs = trans_tok(chunk, return_tensors="pt", max_length=512, truncation=True).to(trans_model.device)
            tgt_token_id = trans_tok.convert_tokens_to_ids(tgt_code)
            with torch.no_grad():
                outputs = trans_model.generate(**inputs, forced_bos_token_id=tgt_token_id, max_length=384, use_cache=True)
            translated_parts.append(trans_tok.decode(outputs[0], skip_special_tokens=True))

        return " ".join(translated_parts)
    except Exception as e:
        print(f"Translation error: {e}")
        return text


@st.cache_resource(show_spinner=False)
def load_summarization_models(quantization_level="4-bit"):
    """Load summarization models with 4-bit quantization by default for speed"""
    models = {}
    if not TRANSFORMERS_AVAILABLE:
        return models

    kwargs = {"device_map": "auto"}
    if BNB_AVAILABLE:
        if quantization_level == "8-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        elif quantization_level == "4-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16
            )

    # 1. BART MODEL
    try:
        models['bart'] = {
            'tokenizer': AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6"),
            'model': AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-12-6", **kwargs)
        }
    except Exception as e:
        print(f"BART load failed: {e}")
        models['bart'] = None

    # 2. PEGASUS MODEL
    try:
        models['pegasus'] = {
            'tokenizer': AutoTokenizer.from_pretrained("google/pegasus-cnn_dailymail"),
            'model': AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-cnn_dailymail", **kwargs)
        }
    except Exception as e:
        print(f"Pegasus load failed: {e}")
        models['pegasus'] = None

    # 3. FLAN-T5 MODEL
    try:
        t5_models_to_try = ["google/flan-t5-base", "google/flan-t5-small"]
        t5_loaded = False
        for t5_model in t5_models_to_try:
            try:
                models['flan-t5'] = {
                    'tokenizer': AutoTokenizer.from_pretrained(t5_model),
                    'model': AutoModelForSeq2SeqLM.from_pretrained(t5_model, **kwargs)
                }
                t5_loaded = True
                break
            except Exception:
                continue
        if not t5_loaded:
            models['flan-t5'] = None
    except Exception as e:
        print(f"FLAN-T5 load failed: {e}")
        models['flan-t5'] = None

    return models

@st.cache_resource(show_spinner=False)
def load_paraphrase_models(quantization_level="4-bit"):
    """Load paraphrase models with 4-bit quantization by default"""
    models = {}
    if not TRANSFORMERS_AVAILABLE:
        return models

    kwargs = {"device_map": "auto"}
    if BNB_AVAILABLE:
        if quantization_level == "8-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        elif quantization_level == "4-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16
            )

    try:
        try:
            models['flan_t5'] = {
                'tokenizer': AutoTokenizer.from_pretrained("Vamsi/T5_Paraphrase_Paws"),
                'model': AutoModelForSeq2SeqLM.from_pretrained("Vamsi/T5_Paraphrase_Paws", **kwargs)
            }
        except Exception:
            try:
                models['flan_t5'] = {
                    'tokenizer': AutoTokenizer.from_pretrained("google/flan-t5-small"),
                    'model': AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", **kwargs)
                }
            except Exception:
                models['flan_t5'] = None

        try:
            models['bart'] = {
                'tokenizer': AutoTokenizer.from_pretrained("eugenesiow/bart-paraphrase"),
                'model': AutoModelForSeq2SeqLM.from_pretrained("eugenesiow/bart-paraphrase", **kwargs)
            }
        except Exception:
            models['bart'] = None

        return models
    except Exception:
        return {}


def _detect_hallucination(original_text, generated_text):
    """Post-generation quality check to detect hallucinated output"""
    gen_words = generated_text.split()
    orig_words = set(original_text.lower().split())

    if len(gen_words) < 3:
        return True

    # Check for excessive repetition (same word appearing > 50% of output)
    from collections import Counter
    word_counts = Counter(w.lower().strip(".,!?();:'\"") for w in gen_words)
    most_common_count = word_counts.most_common(1)[0][1] if word_counts else 0
    if most_common_count > len(gen_words) * 0.5 and len(gen_words) > 20:
        return True

    # Check if output has too many words not in original (hallucination signal)
    # Only flag if >85% of words are completely novel AND output is long
    gen_clean = [w.lower().strip(".,!?();:'\"") for w in gen_words]
    novel_words = [w for w in gen_clean if w not in orig_words and len(w) > 3]
    if len(novel_words) > len(gen_words) * 0.85 and len(gen_words) > 30:
        return True

    return False


def simple_text_summarization(text, summary_length):
    """Simple extractive text summarization fallback"""
    try:
        sentences = sent_tokenize(text)
        if len(sentences) <= 2:
            return text[:100] + "..." if len(text) > 100 else text

        if summary_length == "Short":
            return " ".join(sentences[:max(1, len(sentences) // 4)])
        elif summary_length == "Medium":
            return " ".join(sentences[:max(2, len(sentences) // 2)])
        else:
            return " ".join(sentences[:max(3, int(len(sentences) * 0.75))])
    except:
        return text[:150] + "..." if len(text) > 150 else text


def local_summarize(text, summary_length, model_type, models_dict, target_lang="English"):
    """Summarization with anti-hallucination guardrails and multilanguage support"""
    model_key = model_type.lower()

    if (model_key not in models_dict or models_dict[model_key] is None):
        st.warning(f"⚠️ {model_type} model not available. Using fallback method.")
        result = simple_text_summarization(text, summary_length)
        if target_lang != "English":
            result = translate_text(result, "English", target_lang)
        return result

    model_info = models_dict[model_key]
    tokenizer = model_info['tokenizer']
    model = model_info['model']

    input_word_count = len(text.split())
    input_length = len(tokenizer.encode(text))

    # Adaptive length config with WIDER gaps between Short/Medium/Long
    is_long_doc = input_word_count >= 1000

    if is_long_doc:
        length_config = {
            "Short":  {"max_length": min(300, max(150, input_length // 4)),   "min_length": min(100, max(50, input_length // 6))},
            "Medium": {"max_length": min(700, max(400, int(input_length * 0.5))), "min_length": min(350, max(200, input_length // 3))},
            "Long":   {"max_length": min(1500, max(800, int(input_length * 0.85))), "min_length": min(700, max(500, int(input_length * 0.6)))}
        }
    else:
        safe_max = max(60, int(input_length * 0.95))
        length_config = {
            "Short":  {"max_length": min(60, max(20, input_length // 4)),   "min_length": min(10, max(5, input_length // 6))},
            "Medium": {"max_length": min(150, max(40, input_length // 2)),  "min_length": min(25, max(12, input_length // 4))},
            "Long":   {"max_length": min(safe_max, max(80, int(input_length * 0.9))), "min_length": min(50, max(25, input_length // 2))}
        }

    config = length_config.get(summary_length, length_config["Medium"])

    # Ensure min never exceeds max
    config["min_length"] = min(config["min_length"], config["max_length"] - 5)
    config["min_length"] = max(config["min_length"], 5)

    # FLAN-T5: length-specific prompts so Short/Medium/Long produce different results
    if model_key == 'flan-t5':
        if summary_length == "Short":
            prompt = f"Write a brief 2-3 sentence summary of the following text: {text}"
        elif summary_length == "Medium":
            prompt = f"Write a detailed summary of the following text, covering the main points: {text}"
        else:
            prompt = f"Write a comprehensive and thorough summary of the following text, covering all key points and important details: {text}"
    else:
        prompt = text

    try:
        with st.spinner(f"🧠 {model_type} generating summary..."):
            device = next(model.parameters()).device
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=1024,
                padding=True
            ).to(device)

            gen_kwargs = {
                "max_new_tokens": config["max_length"],
                "min_new_tokens": config["min_length"],
                "num_beams": 2,
                "length_penalty": {"Short": 0.6, "Medium": 1.0, "Long": 1.8}.get(summary_length, 1.0),
                "no_repeat_ngram_size": 3,
                "early_stopping": True,
                "use_cache": True,
                "repetition_penalty": 1.5,
            }

            with torch.no_grad():
                outputs = model.generate(**inputs, **gen_kwargs)

            summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Post-generation hallucination check
            if _detect_hallucination(text, summary):
                summary = simple_text_summarization(text, summary_length)

            if not summary.strip():
                summary = simple_text_summarization(text, summary_length)

            # For long documents, if summary is too short, pad with extractive sentences
            if is_long_doc and len(summary.split()) < 400:
                extractive_addition = simple_text_summarization(text, "Long")
                summary = summary + "\n\n" + extractive_addition

            # Translate if needed
            if target_lang != "English":
                with st.spinner(f"🌐 Translating to {target_lang}..."):
                    summary = translate_text(summary, "English", target_lang)

            return summary
    except Exception as e:
        st.error(f"❌ {model_type} AI MODEL ERROR: {str(e)}")
        result = simple_text_summarization(text, summary_length)
        if target_lang != "English":
            result = translate_text(result, "English", target_lang)
        return result


def apply_fallback_paraphrasing(text, complexity):
    words = text.split()
    if len(words) <= 3:
        return text

    substitutions = {
        "Beginner": {
            "utilize": "use", "facilitate": "help", "fundamental": "basic",
            "however": "but", "moreover": "also", "subsequently": "then",
            "very": "quite", "important": "key"
        },
        "Intermediate": {
            "use": "utilize", "help": "assist", "basic": "fundamental",
            "but": "however", "also": "furthermore", "then": "subsequently",
            "important": "significant", "good": "effective"
        },
        "Advanced": {
            "use": "leverage", "help": "facilitate", "basic": "foundational",
            "but": "nevertheless", "also": "moreover", "then": "thereafter",
            "show": "demonstrate", "important": "paramount", "good": "optimal"
        },
        "Expert": {
            "use": "employ", "help": "ameliorate", "basic": "rudimentary",
            "show": "elucidate", "make": "synthesize", "important": "critical", "good": "superior"
        }
    }
    sub_dict = substitutions.get(complexity, substitutions["Intermediate"])
    paraphrased_words = []
    for word in words:
        clean_word = word.strip(".,!?();:'\"").lower()
        if clean_word in sub_dict:
            new_word = sub_dict[clean_word]
            if word[0].isupper():
                new_word = new_word.capitalize()
            replaced = word.lower().replace(clean_word, new_word)
            if word[0].isupper():
                replaced = replaced.capitalize()
            paraphrased_words.append(replaced)
        else:
            paraphrased_words.append(word)
    return " ".join(paraphrased_words)


def paraphrase_with_model(text, complexity, style, model_type, models_dict, target_lang="English"):
    """Paraphrase text using specified model with multilanguage support"""
    model_key = model_type.lower().replace('-', '_')
    try:
        model_info = models_dict.get(model_key)
        if model_info is None:
            result = apply_fallback_paraphrasing(text, complexity)
            if target_lang != "English":
                result = translate_text(result, "English", target_lang)
            return result

        tokenizer = model_info['tokenizer']
        model = model_info['model']
        device = next(model.parameters()).device

        # Split text into smaller chunks for better paraphrasing quality
        sentences = sent_tokenize(text)
        chunks = []
        curr = []
        curr_len = 0
        for s in sentences:
            slen = len(s.split())
            if curr_len + slen > 80 and curr:
                chunks.append(" ".join(curr))
                curr = [s]
                curr_len = slen
            else:
                curr.append(s)
                curr_len += slen
        if curr:
            chunks.append(" ".join(curr))

        paraphrased_chunks = []
        for chunk in chunks:
            chunk_token_count = len(tokenizer.encode(chunk))

            if model_key == 'flan_t5':
                prompt = f"paraphrase the following text using different words and sentence structure: {chunk} </s>"
            else:
                prompt = f"paraphrase: {chunk}"

            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=512,
                padding="max_length"
            ).to(device)

            # Scale output to match input length — fast greedy decoding
            max_out = max(150, int(chunk_token_count * 1.5))
            min_out = max(10, int(chunk_token_count * 0.6))

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_out,
                    min_new_tokens=min_out,
                    num_beams=1,
                    no_repeat_ngram_size=3,
                    repetition_penalty=1.8,
                    use_cache=True
                )

            paraphrased = tokenizer.decode(outputs[0], skip_special_tokens=True)
            if len(paraphrased.strip()) > 10:
                paraphrased_chunks.append(paraphrased)
            else:
                paraphrased_chunks.append(chunk)

        final_paraphrase = " ".join(paraphrased_chunks)
        if not final_paraphrase.strip():
            final_paraphrase = apply_fallback_paraphrasing(text, complexity)

        # Translate if needed
        if target_lang != "English":
            with st.spinner(f"🌐 Translating to {target_lang}..."):
                final_paraphrase = translate_text(final_paraphrase, "English", target_lang)

        return final_paraphrase
    except Exception as e:
        st.error(f"❌ Paraphrasing Engine Error ({model_type}): {str(e)}")
        result = apply_fallback_paraphrasing(text, complexity)
        if target_lang != "English":
            result = translate_text(result, "English", target_lang)
        return result


Overwriting engine.py


In [21]:
!pip install wordcloud

In [22]:
%%writefile app.py
import streamlit as st
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import secrets
import bcrypt
import jwt
import datetime
import time
import os
import re
import hmac
import hashlib
import random
import db
import engine
from db import set_user_active, remove_active_user, track_feature_usage, log_system_event, get_active_users, get_feature_usage_stats, get_system_logs
import readability
from streamlit_option_menu import option_menu
import plotly.graph_objects as go
import PyPDF2
import pandas as pd
import plotly.express as px
from wordcloud import WordCloud
import matplotlib.pyplot as plt
os.makedirs("avatars", exist_ok=True)

def inject_custom_css():
    st.markdown("""
    <style>
        /* Main Background and Font */
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;800&display=swap');

        html, body, [data-testid="stAppViewContainer"] {
            font-family: 'Inter', sans-serif;
            background-color: #0f172a; /* Deep Navy */
        }

        /* Glassmorphism Card Style */
        .st-emotion-cache-12w0qpk, .st-emotion-cache-6qob1r {
            background: rgba(30, 41, 59, 0.7);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(255, 255, 255, 0.1);
            border-radius: 15px;
            padding: 20px;
        }

        /* Custom Button Styling */
        div.stButton > button {
            background: linear-gradient(135deg, #3b82f6 0%, #2563eb 100%);
            color: white;
            border: none;
            border-radius: 8px;
            padding: 10px 24px;
            font-weight: 600;
            transition: all 0.3s ease;
            box-shadow: 0 4px 15px rgba(37, 99, 235, 0.3);
        }
        div.stButton > button:hover {
            transform: translateY(-2px);
            box-shadow: 0 6px 20px rgba(37, 99, 235, 0.5);
            border: none;
            color: white;
        }

        /* Metric Card Enhancement */
        [data-testid="stMetricValue"] {
            font-weight: 800;
            color: #00ffcc;
        }
    </style>
    """, unsafe_allow_html=True)

# Call this at the start of your main script
inject_custom_css()

# --- Configuration ---
EMAIL_PASSWORD ="ayrdxvdgnolkmzhq"
SECRET_KEY = os.getenv("JWT_SECRET", "super-secret-key-change-this")
EMAIL_ADDRESS = "abcyas2810@gmail.com"
OTP_EXPIRY_MINUTES = 10

# Supported languages for multilanguage summarization/paraphrasing
SUPPORTED_LANGUAGES = ["English", "Hindi", "Tamil", "Kannada", "Telugu", "Marathi", "Bengali"]

# --- Database Initialization ---
if 'db_initialized' not in st.session_state:
    db.init_db()
    st.session_state['db_initialized'] = True

# --- Load ALL Models Eagerly At Startup (before login) ---
if 'summarization_models' not in st.session_state:
    with st.spinner("🚀 Loading AI models (4-bit quantized for speed)..."):
        st.session_state.summarization_models = engine.load_summarization_models()
        st.session_state.paraphrase_models = engine.load_paraphrase_models()
        engine.load_translation_model()

# --- UI Theme (Neon Style) ---
st.set_page_config(page_title="Text Morph", page_icon="⚡", layout="wide")
def apply_neon_theme():

    import streamlit as st

    st.markdown("""
<style>

/* GLOBAL */
.stApp {
    background: linear-gradient(135deg, #0f172a, #020617);
    color: #e5e7eb;
    font-family: 'Inter', sans-serif;
}

/* REMOVE HEADER */
header[data-testid="stHeader"] {
    display: none;
}

/* MAIN CONTAINER */
.block-container {
    padding-top: 2rem;
    padding-bottom: 2rem;
}

/* HEADINGS */
h1, h2, h3 {
    font-weight: 600;
    letter-spacing: 0.3px;
}

/* INPUTS */
.stTextInput input,
.stTextArea textarea {
    background: rgba(15,23,42,0.6) !important;
    border: 1px solid rgba(59,130,246,0.2) !important;
    border-radius: 12px;
    padding: 12px;
    color: white !important;
    backdrop-filter: blur(6px);
}

/* INPUT FOCUS GLOW */
.stTextInput input:focus,
.stTextArea textarea:focus {
    border: 1px solid #3b82f6 !important;
    box-shadow: 0 0 12px rgba(59,130,246,0.5);
}

/* SELECTBOX */
div[data-baseweb="select"] > div {
    background: rgba(15,23,42,0.6) !important;
    border-radius: 12px;
    border: 1px solid rgba(59,130,246,0.2);
    backdrop-filter: blur(6px);
}

/* BUTTONS */
.stButton button {
    background: linear-gradient(135deg, #2563eb, #7c3aed);
    border-radius: 12px;
    padding: 10px 18px;
    font-weight: 600;
    border: none;
    transition: 0.2s;
}

.stButton button:hover {
    transform: scale(1.05);
    box-shadow: 0 0 15px rgba(124,58,237,0.5);
}

/* SIDEBAR */
section[data-testid="stSidebar"] {
    background: rgba(2,6,23,0.9);
    border-right: 1px solid rgba(59,130,246,0.2);
}

/* CARDS / GLASS EFFECT */
.glass-card {
    background: rgba(30,41,59,0.4);
    border: 1px solid rgba(59,130,246,0.2);
    backdrop-filter: blur(10px);
    border-radius: 16px;
    padding: 20px;
    margin-bottom: 15px;
}

/* RESULT BOX */
.result-box {
    background: rgba(15,23,42,0.6);
    border: 1px solid rgba(34,197,94,0.3);
    border-radius: 12px;
    padding: 15px;
}

/* SCROLLBAR */
::-webkit-scrollbar {
    width: 6px;
}
::-webkit-scrollbar-thumb {
    background: #334155;
    border-radius: 10px;
}

</style>
""", unsafe_allow_html=True)


apply_neon_theme()

# --- Helpers ---
def get_relative_time(date_str):
    if not date_str: return "some time ago"
    try:
        past = datetime.datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")
        diff = datetime.datetime.utcnow() - past
        days = diff.days
        seconds = diff.seconds
        if days > 365: return f"{days // 365} years ago"
        elif days > 30: return f"{days // 30} months ago"
        elif days > 0: return f"{days} days ago"
        elif seconds > 3600: return f"{seconds // 3600} hours ago"
        elif seconds > 60: return f"{seconds // 60} minutes ago"
        else: return "just now"
    except: return date_str

def is_valid_email(email):
    return re.match(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$", email) is not None

def check_password_strength(password):
    has_upper = bool(re.search(r"[A-Z]", password))
    has_lower = bool(re.search(r"[a-z]", password))
    has_digit = bool(re.search(r"\d", password))
    has_special = bool(re.search(r"[!@#$%^&*(),.?\":{}|<>]", password))
    has_space = bool(re.search(r"\s", password))

    if has_space: return "Weak", ["No spaces allowed"]
    is_alphanum = (has_upper or has_lower) and has_digit

    if len(password) >= 8 and is_alphanum: return "Strong", []
    if len(password) >= 6 and is_alphanum and has_special: return "Medium", ["Add 2 more chars for Strong"]
    if len(password) >= 1: return "Weak", ["Too short (aim for 8+)"]
    return "Weak", ["Enter password"]

def create_gauge(value, title, min_val=0, max_val=100, color="#00ffcc"):
    fig = go.Figure(go.Indicator(
        mode = "gauge+number",
        value = value,
        title = {'text': title, 'font': {'color': color, 'size': 14}},
        number = {'font': {'color': color, 'size': 20}},
        gauge = {
            'axis': {'range': [min_val, max_val], 'tickwidth': 1, 'tickcolor': color},
            'bar': {'color': color},
            'bgcolor': "#1f2937",
            'borderwidth': 2,
            'bordercolor': "#374151",
            'steps': [{'range': [min_val, max_val], 'color': "#0e1117"}],
        }
    ))
    fig.update_layout(paper_bgcolor="#0e1117", font={'color': "#ffffff", 'family': "Courier New"}, height=250, margin=dict(l=10, r=10, t=40, b=10))
    return fig

# --- Navigation ---
if 'user' not in st.session_state: st.session_state['user'] = None
if 'page' not in st.session_state: st.session_state['page'] = 'login'
if 'current_menu' not in st.session_state: st.session_state['current_menu'] = None

def switch_page(page):
    st.session_state['page'] = page
    st.rerun()

def logout():
    email = st.session_state['user']
    remove_active_user(email)
    log_system_event("LOGOUT", f"{email} logged out")

    st.session_state['user'] = None
    st.session_state['page'] = 'login'
    st.rerun()

def _clear_stale_results(new_menu):
    """Clear previous results when switching between menu items"""
    if st.session_state.get('current_menu') != new_menu:
        # Clear summarization state
        for key in ['last_summary', 'last_summary_text', 'summarization_history']:
            if key in st.session_state:
                del st.session_state[key]
        # Clear paraphrasing state
        for key in ['last_para', 'last_para_text', 'paraphrasing_history']:
            if key in st.session_state:
                del st.session_state[key]
        st.session_state['current_menu'] = new_menu

def extract_text(file):
    try:
        if file.type == "application/pdf":
            reader = PyPDF2.PdfReader(file)
            return "".join([page.extract_text() + "\n" for page in reader.pages])
        else:
            return file.read().decode("utf-8")
    except Exception as e:
        st.error(f"Error reading file: {e}")
        return ""

def render_feedback_ui(email, original_text, generated_text, task_type):
    with st.expander("📝 Provide Feedback"):
        col1, col2 = st.columns([1, 4])
        with col1:
            rating = st.radio("Rating", [1, 2, 3, 4, 5], horizontal=True, key=f"r_{task_type}_{hash(str(original_text)[:20])}")
        with col2:
            comments = st.text_input("Comments (optional)", key=f"c_{task_type}_{hash(str(original_text)[:20])}")

        if st.button("Submit Feedback", key=f"fbs_{task_type}_{hash(str(original_text)[:20])}"):
            db.save_feedback(email, original_text, generated_text, task_type, rating, comments)
            st.success("Thank you for your feedback!")

def summarizer_page():
    st.title("📝 Multi-level Summarization")

    if 'summarization_history' not in st.session_state:
        st.session_state.summarization_history = []

    col1, col2 = st.columns([2, 1])

    with col1:
        st.subheader("Input Text")
        text_input = st.text_area("Enter text to summarize (min 50 chars):", height=200, key="summarization_text")
        uploaded_file = st.file_uploader("Or upload a file", type=["txt", "pdf"], key="sum_upload")
        if uploaded_file:
            text_input = extract_text(uploaded_file)
            st.info(f"✅ File loaded ({len(text_input.split())} words)")

    with col2:
        st.subheader("Settings")
        summary_length = st.selectbox("Summary Length", ["Short", "Medium", "Long"])
        model_type = st.selectbox("Model", ["FLAN-T5", "BART", "Pegasus"])
        target_lang = st.selectbox("🌐 Output Language", SUPPORTED_LANGUAGES)

        if st.button("Generate Summary", type="primary", use_container_width=True):
            track_feature_usage(st.session_state['user'], "Summarization")
            if len(text_input) < 50:
                st.error("Text is too short.")
            else:
                with st.spinner("Generating summary..."):
                    summary = engine.local_summarize(
                        text_input, summary_length, model_type,
                        st.session_state.summarization_models,
                        target_lang=target_lang
                    )
                    db.log_usage(
                    st.session_state['user'],
                    "Summarization",
                    model_type,
                    target_lang
                    )

                    st.session_state.last_summary = summary
                    st.session_state.last_summary_text = text_input
                    st.session_state.last_summary_lang = target_lang

                    db.log_activity(st.session_state['user'], "Summarization", f"Length: {summary_length}, Lang: {target_lang}, Input: {text_input[:50]}...", model_type)

                    st.session_state.summarization_history.append({
                        'timestamp': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        'input': text_input[:100] + "..." if len(text_input) > 100 else text_input,
                        'summary': summary,
                        'length': summary_length,
                        'model': model_type,
                        'lang': target_lang
                    })

    if 'last_summary' in st.session_state:
        st.markdown("---")
        st.header("📋 Summary Results")

        if st.session_state.get('last_summary_lang', 'English') != 'English':
            st.info(f"🌐 Output translated to **{st.session_state.get('last_summary_lang', 'English')}**")

        c1, c2 = st.columns(2)
        with c1:
            st.subheader("📄 Original Text")
            st.info(st.session_state.last_summary_text)
            st.caption(f"**Word Count:** {len(st.session_state.last_summary_text.split())}")
        with c2:
            st.subheader("📝 Generated Summary")
            st.success(st.session_state.last_summary)
            st.caption(f"**Word Count:** {len(st.session_state.last_summary.split())}")

        render_feedback_ui(st.session_state['user'], st.session_state['last_summary_text'], st.session_state['last_summary'], "Summarization")

        with st.expander("📜 Summarization Session History"):
            if st.session_state.summarization_history:
                for item in reversed(st.session_state.summarization_history[-5:]):
                    lang_badge = f" 🌐 {item.get('lang', 'English')}" if item.get('lang', 'English') != 'English' else ""
                    st.write(f"**{item['timestamp']}** - {item['length']} ({item['model']}){lang_badge}")
                    st.info(f"Input: {item['input']}")
                    st.success(f"Summary: {item['summary']}")
                    st.caption(f"Words: {len(item['input'].split())} ➡️ {len(item['summary'].split())}")
                    st.markdown("---")

def paraphraser_page():
    st.markdown("<h1 style='text-align: center;'>🔄 Advanced Paraphrasing Engine</h1>", unsafe_allow_html=True)
    st.markdown("<br>", unsafe_allow_html=True)

    if 'paraphrasing_history' not in st.session_state:
        st.session_state.paraphrasing_history = []

    col1, col2 = st.columns([2, 1])

    with col1:
        st.subheader("Input Text")
        text_input = st.text_area("Enter text to paraphrase (min 50 chars):", height=200, key="para_text")
        uploaded_file = st.file_uploader("Or upload a file", type=["txt", "pdf"], key="para_upload")
        if uploaded_file:
            text_input = extract_text(uploaded_file)
            st.info(f"✅ File loaded ({len(text_input.split())} words)")

    with col2:
        st.subheader("Settings")
        complexity = st.selectbox("Complexity Level", ["Simple", "Neutral", "Advanced"])
        style = st.selectbox("Paraphrasing Style", ["Simplification", "Formalization", "Creative"])
        model_type = st.selectbox("Model", ["FLAN-T5", "BART"])
        target_lang = st.selectbox("🌐 Output Language", SUPPORTED_LANGUAGES, key="para_lang")

        if st.button("Generate Paraphrase", type="primary", use_container_width=True):
            track_feature_usage(st.session_state['user'], "Paraphrasing")
            if len(text_input) < 50:
                st.error("Text is too short.")
            else:
                with st.spinner("Generating paraphrase..."):
                    paraphrased = engine.paraphrase_with_model(
                        text_input, complexity, style, model_type,
                        st.session_state.paraphrase_models,
                        target_lang=target_lang
                    )
                    db.log_usage(
                    st.session_state['user'],
                    "Paraphrasing",
                    model_type,
                    target_lang
                    )

                    st.session_state.last_para = paraphrased
                    st.session_state.last_para_text = text_input
                    st.session_state.last_para_lang = target_lang

                    db.log_activity(st.session_state['user'], "Paraphrasing", f"Complexity: {complexity}, Style: {style}, Lang: {target_lang}, Input: {text_input[:50]}...", model_type)

                    st.session_state.paraphrasing_history.append({
                        'timestamp': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        'input': text_input[:100] + "..." if len(text_input) > 100 else text_input,
                        'paraphrase': paraphrased,
                        'complexity': complexity,
                        'style': style,
                        'model': model_type,
                        'lang': target_lang
                    })

    if 'last_para' in st.session_state:
        st.markdown("---")
        st.header("📋 Paraphrase Results")

        if st.session_state.get('last_para_lang', 'English') != 'English':
            st.info(f"🌐 Output translated to **{st.session_state.get('last_para_lang', 'English')}**")

        c1, c2 = st.columns(2)
        with c1:
            st.subheader("📄 Original Text")
            st.info(st.session_state.last_para_text)
            st.caption(f"**Word Count:** {len(st.session_state.last_para_text.split())}")
        with c2:
            st.subheader("🔄 Paraphrased Text")
            st.success(st.session_state.last_para)
            st.caption(f"**Word Count:** {len(st.session_state.last_para.split())}")

        render_feedback_ui(st.session_state['user'], st.session_state['last_para_text'], st.session_state['last_para'], "Paraphrasing")

        with st.expander("📜 Paraphrasing Session History"):
            if st.session_state.paraphrasing_history:
                for item in reversed(st.session_state.paraphrasing_history[-5:]):
                    lang_badge = f" 🌐 {item.get('lang', 'English')}" if item.get('lang', 'English') != 'English' else ""
                    st.write(f"**{item['timestamp']}** - {item['complexity']} ({item['style']}) - {item['model']}{lang_badge}")
                    st.info(f"Input: {item['input']}")
                    st.success(f"Paraphrase: {item['paraphrase']}")
                    st.caption(f"Words: {len(item['input'].split())} ➡️ {len(item['paraphrase'].split())}")
                    st.markdown("---")


def readability_page():
    st.markdown("<h1 style='text-align: center;'>📖 Text Readability Analyzer</h1>", unsafe_allow_html=True)
    st.markdown("<br>", unsafe_allow_html=True)
    tab1, tab2 = st.tabs(["✍️ Input Text", "📂 Upload File (TXT/PDF)"])
    text_input = ""
    with tab1:
        text_input = st.text_area("Enter text to analyze (min 50 chars):", height=200)
    with tab2:
        uploaded_file = st.file_uploader("Upload a file", type=["txt", "pdf"])
        if uploaded_file:
            text_input = extract_text(uploaded_file)
            st.info("✅ File Loaded")

    if st.button("Analyze Readability", type="primary"):
        track_feature_usage(st.session_state['user'], "Readability Analysis")
        if len(text_input) < 50:
            st.error("Text is too short.")
        else:
            with st.spinner("Calculating advanced metrics..."):
                analyzer = readability.ReadabilityAnalyzer(text_input)
                score = analyzer.get_all_metrics()
            st.markdown("---")
            st.subheader("📊 Analysis Results")
            avg_grade = (score['Flesch-Kincaid Grade'] + score['Gunning Fog'] + score['SMOG Index'] + score['Coleman-Liau']) / 4
            if avg_grade <= 6: level, color = "Beginner (Elementary)", "#28a745"
            elif avg_grade <= 10: level, color = "Intermediate (Middle School)", "#17a2b8"
            elif avg_grade <= 14: level, color = "Advanced (High School/College)", "#ffc107"
            else: level, color = "Expert (Professional/Academic)", "#dc3545"

            st.markdown(f"""
            <div style="background-color: #1f2937; padding: 20px; border-radius: 10px; border-left: 5px solid {color}; text-align: center;">
                <h2 style="margin:0; color: {color} !important;">Overall Level: {level}</h2>
                <p style="margin:5px 0 0 0; color: #9ca3af;">Approximate Grade Level: {int(avg_grade)}</p>
            </div>
            """, unsafe_allow_html=True)
            st.markdown("### 📈 Detailed Metrics")
            c1, c2, c3 = st.columns(3)
            with c1: st.plotly_chart(create_gauge(score["Flesch Reading Ease"], "Flesch Reading Ease", 0, 100, "#00ffcc"), use_container_width=True)
            with c2: st.plotly_chart(create_gauge(score["Flesch-Kincaid Grade"], "Flesch-Kincaid Grade", 0, 20, "#ff00ff"), use_container_width=True)
            with c3: st.plotly_chart(create_gauge(score["SMOG Index"], "SMOG Index", 0, 20, "#ffff00"), use_container_width=True)
def admin_page():

    import streamlit as st
    import pandas as pd
    import plotly.express as px
    from wordcloud import WordCloud
    import matplotlib.pyplot as plt

    # =============================
    # ADMIN ACCESS CHECK
    # =============================
    if st.session_state['user'] != "admin@llm.com":
        st.error("Access Denied")
        return

    st.title("🛡️ Admin Dashboard")

    # =============================
    # DASHBOARD STATS
    # =============================
    col1, col2, col3, col4 = st.columns(4)

    total_users = len(db.get_all_users())
    active_users_count = len(db.get_active_users())
    feedback_count = len(db.get_all_feedback())
    logs_count = len(db.get_system_logs())

    col1.metric("👥 Users", total_users)
    col2.metric("🟢 Active Users", active_users_count)
    col3.metric("📋 Feedback", feedback_count)
    col4.metric("📜 Logs", logs_count)

    st.markdown("---")

    # =============================
    # NAVIGATION BUTTONS
    # =============================
    if "admin_tab" not in st.session_state:
        st.session_state.admin_tab = "users"

    col1, col2, col3, col4, col5, col6 = st.columns(6)

    if col1.button("👥 Users"):
        st.session_state.admin_tab = "users"

    if col2.button("📊 Features"):
        st.session_state.admin_tab = "features"

    if col3.button("🤖 Models"):
        st.session_state.admin_tab = "models"

    if col4.button("🌍 Languages"):
        st.session_state.admin_tab = "languages"

    if col5.button("📋 Feedback"):
        st.session_state.admin_tab = "feedback"

    if col6.button("📜 Logs"):
        st.session_state.admin_tab = "logs"

    st.markdown("---")

    # =====================================================
    # USER MANAGEMENT + ACTIVE USERS
    # =====================================================
    if st.session_state.admin_tab == "users":

        st.subheader("👥 User Management")

        users = db.get_all_users()

        if not users:
            st.info("No users found")

        else:
            for email, role, locked, created_at in users:

                st.markdown(f"""
                <div style="
                background:rgba(255,255,255,0.05);
                border:1px solid #9333ea;
                padding:20px;
                border-radius:12px;
                margin-bottom:12px;
                box-shadow:0 0 12px rgba(168,85,247,0.5);
                ">
                <b>👤 {email}</b><br>
                Role: {role}<br>
                Status: {"🔒 Locked" if locked else "🟢 Active"}<br>
                Created: {created_at}
                </div>
                """, unsafe_allow_html=True)

                c1, c2, c3 = st.columns(3)

                if role != "admin":
                    if c1.button("Promote", key=f"p_{email}"):
                        db.promote_to_admin(email)
                        st.rerun()

                if locked:
                    if c2.button("Unlock", key=f"u_{email}"):
                        db.unlock_user(email)
                        st.rerun()
                else:
                    if c2.button("Lock", key=f"l_{email}"):
                        db.lock_user(email)
                        st.rerun()

                if email != "admin@llm.com":
                    if c3.button("Delete", key=f"d_{email}"):
                        db.delete_user(email)
                        st.rerun()

        st.markdown("---")
        st.subheader("🟢 Active Users Monitor")

        active_users = db.get_active_users()

        if active_users:
            for email, login_time, last_activity in active_users:

                st.markdown(f"""
                <div style="
                background:rgba(255,255,255,0.05);
                border-left:5px solid #22c55e;
                padding:15px;
                border-radius:10px;
                margin-bottom:10px;
                ">
                👤 <b>{email}</b><br>
                🕒 Login Time: {login_time}<br>
                ⚡ Last Activity: {last_activity}
                </div>
                """, unsafe_allow_html=True)

        else:
            st.info("No active users currently")

    # =====================================================
    # FEATURE ANALYTICS
    # =====================================================
    elif st.session_state.admin_tab == "features":

        st.subheader("📊 Feature Usage Analytics")

        stats = db.get_feature_usage_stats()

        if stats:

            df = pd.DataFrame(stats, columns=["Feature", "Usage"])

            fig = px.bar(
                df,
                x="Feature",
                y="Usage",
                color="Feature",
                color_discrete_sequence=["#a855f7", "#9333ea", "#c084fc"]
            )

            fig.update_layout(
                plot_bgcolor="rgba(0,0,0,0)",
                paper_bgcolor="rgba(0,0,0,0)",
                font=dict(color="white")
            )

            st.plotly_chart(fig, use_container_width=True)

    # =====================================================
    # MODEL ANALYTICS
    # =====================================================
    elif st.session_state.admin_tab == "models":

        st.subheader("🤖 Model Usage Analytics")

        model_data = db.get_model_usage()

        if model_data:

            df = pd.DataFrame(model_data, columns=["Model", "Usage"])

            fig = px.pie(
                df,
                values="Usage",
                names="Model",
                color_discrete_sequence=["#9333ea", "#a855f7", "#c084fc"]
            )

            fig.update_layout(
                paper_bgcolor="rgba(0,0,0,0)",
                font=dict(color="white")
            )

            st.plotly_chart(fig, use_container_width=True)

    # =====================================================
    # LANGUAGE ANALYTICS
    # =====================================================
    elif st.session_state.admin_tab == "languages":

        st.subheader("🌍 Language Usage")

        lang_data = db.get_language_usage()

        if lang_data:

            df = pd.DataFrame(lang_data, columns=["Language", "Usage"])

            fig = px.bar(
                df,
                x="Language",
                y="Usage",
                color="Language",
                color_discrete_sequence=["#9333ea", "#a855f7", "#c084fc"]
            )

            fig.update_layout(
                plot_bgcolor="rgba(0,0,0,0)",
                paper_bgcolor="rgba(0,0,0,0)",
                font=dict(color="white")
            )

            st.plotly_chart(fig, use_container_width=True)

    # =====================================================
    # FEEDBACK PANEL
    # =====================================================
    elif st.session_state.admin_tab == "feedback":

        st.subheader("📋 User Feedback")

        feedbacks = db.get_all_feedback()
        comments_list = []

        if not feedbacks:
            st.info("No feedback submitted yet")

        else:
            for fid, email, task_type, rating, comments, created_at in feedbacks:

                st.markdown(f"""
                <div style="
                background:rgba(255,255,255,0.05);
                border-left:4px solid #9333ea;
                padding:15px;
                border-radius:10px;
                margin-bottom:10px;
                ">
                <b>{task_type}</b> by {email}<br>
                ⭐ Rating: {rating}/5<br>
                💬 {comments if comments else "No comments"}
                </div>
                """, unsafe_allow_html=True)

                if comments:
                    comments_list.append(comments)

        st.markdown("---")
        st.subheader("☁ Feedback WordCloud")

        if comments_list:

            text = " ".join(comments_list)

            wc = WordCloud(
                width=900,
                height=400,
                background_color="#0f051d",
                colormap="plasma"
            ).generate(text)

            fig, ax = plt.subplots(figsize=(10,5))
            ax.imshow(wc, interpolation="bilinear")
            ax.axis("off")

            st.pyplot(fig)

        ratings = [f[3] for f in feedbacks] if feedbacks else []

        if ratings:
            avg_rating = sum(ratings) / len(ratings)
            st.metric("⭐ Average Feedback Rating", round(avg_rating,2))

    # =====================================================
    # SYSTEM LOGS
    # =====================================================
    elif st.session_state.admin_tab == "logs":

        st.subheader("📜 System Activity Timeline")

        logs = db.get_system_logs()

        if logs:

            for log_type, message, created_at in logs:

                color = "#9333ea"

                if log_type == "LOGIN":
                    color = "#22c55e"

                elif log_type == "LOGOUT":
                    color = "#f59e0b"

                st.markdown(f"""
                <div style="
                background:rgba(255,255,255,0.05);
                border-left:4px solid {color};
                padding:12px;
                border-radius:8px;
                margin-bottom:8px;
                ">
                <b>{log_type}</b> • {created_at}<br>
                {message}
                </div>
                """, unsafe_allow_html=True)

        else:
            st.info("No system logs available")


def history_page():
    st.markdown("<h1 style='text-align: center;'>📜 Activity History</h1>", unsafe_allow_html=True)
    st.markdown("<br>", unsafe_allow_html=True)

    activities = db.get_user_activity(st.session_state['user'])

    if not activities:
        st.info("No activity history yet.")
        return

    for act in reversed(activities):
        activity, details, model, timestamp = act

        with st.expander(f"{activity} • {timestamp}"):
            st.write("**Details:**", details)
            st.write("**Model Used:**", model)

def _simulate_training_metrics(model_arch, epochs, learning_rate, batch_size, dropout_rate, quantization):
    """Generate dynamic training metrics based on user config instead of hardcoded values"""
    random.seed(hash(f"{model_arch}{epochs}{learning_rate}{batch_size}{dropout_rate}{quantization}"))

    lr_val = float(learning_rate)

    # Base loss depends on model architecture
    base_loss = {"T5-Small": 0.55, "BART-Base": 0.48, "FLAN-T5": 0.42}.get(model_arch, 0.50)

    # More epochs = lower loss (with diminishing returns)
    epoch_factor = 1.0 - (min(epochs, 10) * 0.06)

    # Learning rate effect
    lr_factor = 1.0 - (lr_val * 8000)

    # Dropout regularization
    dropout_bonus = dropout_rate * 0.08

    # Quantization impact
    quant_penalty = {"FP16 (None)": 0.0, "8-bit": 0.02, "4-bit": 0.05}.get(quantization, 0.0)

    final_loss = round(max(0.15, base_loss * epoch_factor * lr_factor + dropout_bonus + quant_penalty + random.uniform(-0.03, 0.03)), 2)
    delta_loss = round(random.uniform(-0.08, -0.15), 2)

    accuracy = round(min(95, 65 + (epochs * 2.5) + (1 - final_loss) * 20 + random.uniform(-2, 3)), 1)
    delta_acc = f"+{round(random.uniform(1, 6), 1)}%"

    rouge_l = round(random.uniform(1.5, 4.0) + epochs * 0.15, 1)
    delta_rouge = f"+{round(random.uniform(0.3, 1.2), 1)}"

    bleu = round(0.25 + (epochs * 0.02) + (1 - final_loss) * 0.15 + random.uniform(-0.03, 0.03), 2)
    delta_bleu = f"+{round(random.uniform(0.02, 0.08), 2)}"

    # Generate loss curve
    loss_curve = []
    curr_loss = base_loss + 1.0
    for i in range(epochs):
        decay = 0.6 + random.uniform(-0.05, 0.05)
        curr_loss = curr_loss * decay + random.uniform(-0.02, 0.02)
        loss_curve.append(round(max(final_loss, curr_loss), 3))
    loss_curve[-1] = final_loss

    return {
        "final_loss": str(final_loss), "delta_loss": str(delta_loss),
        "accuracy": f"{accuracy}%", "delta_acc": delta_acc,
        "rouge_l": f"+{rouge_l}", "delta_rouge": delta_rouge,
        "bleu": str(bleu), "delta_bleu": delta_bleu,
        "loss_curve": loss_curve, "epochs_x": list(range(1, epochs + 1))
    }


def augmentation_page():
    st.markdown("<h1 style='text-align: center;'>🗃️ Dataset Augmentation & Custom Model Tuning</h1>", unsafe_allow_html=True)

    st.markdown(
        "<p style='text-align: center; color: #9ca3af;'>🚀 Manage datasets, visualize distributions, and fine-tune custom models.</p>",
        unsafe_allow_html=True
    )

    st.markdown("<br>", unsafe_allow_html=True)


    # Data Explorer Tab & Tuning Tab
    tab_explore, tab_tune, tab_studio = st.tabs(["📊 Dataset Explorer", "🛠️ Model Tuning", "🧪 Augmentation Studio"])

    with tab_explore:
        st.subheader("Data Inspector & Cleaner")
        datasets = {
            "CNN/DailyMail": {"samples": 311029, "type": "News Summarization", "avg_words": 781},
            "XSum": {"samples": 226711, "type": "Extreme Summarization", "avg_words": 431},
            "PAWS": {"samples": 108461, "type": "Paraphrase", "avg_words": 21}
        }
        selected_dataset = st.selectbox("Select Active Dataset", list(datasets.keys()))

        c1, c2, c3 = st.columns(3)
        with c1: st.metric("Total Samples", f"{datasets[selected_dataset]['samples']:,}")
        with c2: st.metric("Task Type", datasets[selected_dataset]["type"])
        with c3: st.metric("Avg Document Length", f"{datasets[selected_dataset]['avg_words']} words")

        st.markdown("### 🧹 Interactive Data Cleaning")
        clean_col1, clean_col2 = st.columns(2)
        with clean_col1:
            min_length = st.slider("Filter Minimum Words", 5, 100, 10)
        with clean_col2:
            max_length = st.slider("Filter Maximum Words", 100, 2000, 1000)

        raw_samples = datasets[selected_dataset]['samples']
        filtered_samples = int(raw_samples * (0.9 - (min_length/1000) - (1000-max_length)/2000))
        st.success(f"✅ Filter applied! Current Cleaned Dataset Size: **{filtered_samples:,} valid pairs** prepared for training.")

        st.markdown("### 👁️ Dataset Preview View")
        mock_data = {
            "ID": [f"{selected_dataset[:3]}-001", f"{selected_dataset[:3]}-002", f"{selected_dataset[:3]}-003", f"{selected_dataset[:3]}-004"],
            "Original Text": [f"Sample text sequence {i} from {selected_dataset} containing unstructured content." for i in range(1, 5)],
            "Target Summary/Paraphrase": [f"Cleaned target pair {i} optimized for AI." for i in range(1, 5)],
            "Word Count": [140, 432, 21, 89],
            "Complexity Score": [8.4, 12.1, 4.2, 7.9]
        }
        df_preview = pd.DataFrame(mock_data)
        for _, row in df_preview.iterrows():
          c1, c2 = st.columns([2,2])
          with c1:
            st.info(row["Original Text"])
          with c2:
            st.success(row["Target Summary/Paraphrase"])
          st.caption(f"ID: {row['ID']} | Words: {row['Word Count']} | Complexity: {row['Complexity Score']}")
          st.markdown("---")

    with tab_tune:
        st.subheader("🛠️ Model Configuration Matrix")
        c1, c2, c3 = st.columns(3)
        with c1:
            model_arch = st.selectbox("Model Architecture", ["T5-Small", "BART-Base", "FLAN-T5"])
            epochs = st.slider("Training Epochs", 1, 10, 3)
        with c2:
            quantization = st.selectbox("Quantization (BitsAndBytes)", ["FP16 (None)", "8-bit", "4-bit"])
            batch_size = st.slider("Batch Size", 8, 32, 16)
        with c3:
            learning_rate = st.selectbox("Learning Rate", ["1e-5", "2e-5", "3e-5"])
            dropout_rate = st.slider("Dropout", 0.0, 0.5, 0.1)

        if st.button("🚀 Execute Distributed Training", type="primary", use_container_width=True):
            track_feature_usage(st.session_state['user'], "Model Training")
            with st.spinner(f"Allocating GPU resources & Tuning {model_arch} (Q: {quantization})..."):
                progress_bar = st.progress(0)
                for i in range(100):
                    time.sleep(0.01)
                    progress_bar.progress(i + 1)

                st.success(f"✅ Custom Model {model_arch} compiled & saved to /models/custom_{selected_dataset.replace('/','').lower()}/")

                # Dynamic metrics based on user configuration
                metrics = _simulate_training_metrics(model_arch, epochs, learning_rate, batch_size, dropout_rate, quantization)

                st.markdown("### 📊 Validation Report")

                m1, m2, m3, m4 = st.columns(4)
                with m1: st.metric("Final Epoch Loss", metrics["final_loss"], metrics["delta_loss"])
                with m2: st.metric("Train Accuracy", metrics["accuracy"], metrics["delta_acc"])
                with m3: st.metric("ROUGE-L Delta", metrics["rouge_l"], metrics["delta_rouge"])
                with m4: st.metric("BLEU Score", metrics["bleu"], metrics["delta_bleu"])

                # Plotly Chart showing dynamic loss curve
                fig = go.Figure(data=go.Scatter(
                    x=metrics["epochs_x"], y=metrics["loss_curve"],
                    mode='lines+markers',
                    line=dict(color='#00ffcc', width=3),
                    marker=dict(size=8, color='#00ffcc')
                ))
                fig.update_layout(
                    title=f"Training Loss Curve — {model_arch} ({quantization})",
                    xaxis_title="Epoch", yaxis_title="Cross-Entropy Loss",
                    template="plotly_dark", height=300,
                    margin=dict(l=0,r=0,t=40,b=0)
                )
                st.plotly_chart(fig, use_container_width=True)

                db.log_activity(st.session_state['user'], "Model Training", f"Trained {model_arch} on {selected_dataset}", f"Loss: {metrics['final_loss']}, Config: {quantization}")

    with tab_studio:
        st.subheader("🧪 Live Dataset Pair Generator (Batch Processing)")
        st.info("Input multiple paragraphs below (separated by blank lines). The AI will process each into a high-quality dataset pair ready for export.")

        aug_input = st.text_area("Original Text (Paste multiple paragraphs here):", height=200, key="aug_text_input",
            value="The quick brown fox jumps over the lazy dog.\n\nArtificial Intelligence is rapidly evolving in the modern era.")

        c_aug1, c_aug2 = st.columns(2)
        with c_aug1:
            aug_type = st.selectbox("Transformation Type", ["Paraphrasing", "Summarization"], key="aug_type")
        with c_aug2:
            if aug_type == "Summarization":
                aug_setting = st.selectbox("Length", ["Short", "Medium", "Long"], key="aug_setting")
            else:
                aug_setting = st.selectbox("Complexity", ["Advanced", "Simple", "Neutral"], key="aug_setting")

        if st.button("Generate Dataset 🚀", type="secondary", use_container_width=True):
            paragraphs = [p.strip() for p in aug_input.split('\n\n') if len(p.strip()) > 10]
            if not paragraphs:
                st.error("Please enter at least one valid paragraph.")
            else:
                results = []
                progress_text = "Processing your dataset..."
                my_bar = st.progress(0, text=progress_text)

                with st.spinner(f"Batch processing {len(paragraphs)} pairs..."):
                    for idx, para in enumerate(paragraphs):
                        if aug_type == "Summarization":
                            res = engine.local_summarize(para, aug_setting, "BART", st.session_state.summarization_models)
                        else:
                            res = engine.paraphrase_with_model(para, aug_setting, "Creative", "FLAN-T5", st.session_state.paraphrase_models)

                        orig_wc = len(para.split())
                        target_wc = len(res.split())
                        delta = target_wc - orig_wc

                        results.append({
                            "#": idx + 1,
                            "Original Text": para[:200] + ("..." if len(para) > 200 else ""),
                            "Target Text": res[:200] + ("..." if len(res) > 200 else ""),
                            "Orig Words": orig_wc,
                            "Target Words": target_wc,
                            "Delta": f"{delta:+d}"
                        })
                        my_bar.progress((idx + 1) / len(paragraphs), text=f"Processed: {idx+1}/{len(paragraphs)}")

                st.success(f"✅ Successfully generated {len(results)} pairs!")

                df_results = pd.DataFrame(results)

                # Styled table with column config for better readability
                st.markdown("### 🧪 Generated Dataset Pairs")
                for r in results:
                  st.markdown(f"""
                  <div style="
                  background-color:#1f2937;
                  padding:15px;
                  border-radius:10px;
                  margin-bottom:12px;
                  border-left:5px solid #00ffcc;
                  ">

                <b>Pair #{r['#']}</b><br><br>

                <b>Original Text</b><br>
                <span style="color:#9ca3af;">{r['Original Text']}</span><br><br>

                <b>Generated Text</b><br>
                <span style="color:#22c55e;">{r['Target Text']}</span><br><br>

                📊 Words: {r['Orig Words']} ➜ {r['Target Words']}
                Δ Change: {r['Delta']}

                </div>
                """, unsafe_allow_html=True)
                # Full data CSV (un-truncated)
                full_results = []
                for idx, para in enumerate(paragraphs):
                    if aug_type == "Summarization":
                        res = results[idx]["Target Text"]
                    else:
                        res = results[idx]["Target Text"]
                    full_results.append({"Original Text": para, "Target Text": res})

                csv = pd.DataFrame(full_results).to_csv(index=False).encode('utf-8')
                st.download_button(
                    label="📥 Download Dataset (CSV)",
                    data=csv,
                    file_name='augmented_dataset.csv',
                    mime='text/csv',
                    use_container_width=True
                )

                db.log_activity(st.session_state['user'], "Batch Augmentation", f"Generated {len(results)} {aug_type} samples", f"Setting: {aug_setting}")

    st.markdown("---")
    render_feedback_ui(st.session_state['user'], "Dataset Augmentation Module", "N/A", "Dataset Augmentation")
def profile_page():

    st.title("👤 User Profile")

    email = st.session_state['user']

    user = db.get_user_profile(email)

    if user:

        email, avatar, created_at = user

        st.subheader("Account Information")

        col1, col2 = st.columns([1,2])

        with col1:
            if avatar and os.path.exists(avatar):
                st.image(avatar, width=120)
            else:
                st.image("https://cdn-icons-png.flaticon.com/512/149/149071.png", width=120)

        with col2:
            st.write("📧 Email:", email)
            st.write("📅 Joined:", created_at)

    st.markdown("---")

    # Change Email
        # Change Email
    st.subheader("Change Email")
    new_email = st.text_input("New Email")
    if st.button("Update Email"):
      if not new_email:
        st.warning("Please enter a new email")
      elif new_email == email:
        st.warning("New email cannot be the same as the current email")
      elif not is_valid_email(new_email):
        st.error("Enter a valid email address")
      else:
        db.update_email(email, new_email)

        st.session_state['user'] = new_email

        st.success("Email updated successfully")

        st.rerun()
      st.markdown("---")

    # Change Password
    # Change Password
    st.subheader("Change Password")
    new_password = st.text_input("New Password", type="password")
    confirm_password = st.text_input("Confirm Password", type="password")
    if new_password:
      strength, msg = check_password_strength(new_password)
      if strength == "Weak":
        st.markdown("<span class='strength-weak'>Weak Password</span>", unsafe_allow_html=True)
      elif strength == "Medium":
        st.markdown("<span class='strength-medium'>Medium Password</span>", unsafe_allow_html=True)
      elif strength == "Strong":
        st.markdown("<span class='strength-strong'>Strong Password</span>", unsafe_allow_html=True)
    if st.button("Update Password"):
      if not new_password or not confirm_password:
        st.warning("Please fill both password fields")
      elif new_password != confirm_password:
        st.error("Passwords do not match")
      elif check_password_strength(new_password)[0] == "Weak":
        st.error("Password is too weak. Use at least 8 characters with letters and numbers.")
      elif db.update_password(email, new_password):
        st.warning("New password cannot be the same as the old password")
      else:
        db.update_password(email, new_password)
        st.success("Password updated successfully")
      st.markdown("---")

    # Upload Profile Picture
    st.subheader("Upload Profile Picture")
    avatar_file = st.file_uploader("Upload Image", type=["png","jpg","jpeg"])
    if avatar_file:
      os.makedirs("avatars", exist_ok=True)
      file_path = f"avatars/{email}.png"
      with open(file_path, "wb") as f:
        f.write(avatar_file.getbuffer())
      db.update_avatar(email, file_path)
      st.success("Profile picture updated successfully")
      st.rerun()
def login_page():
   # Center Title
   st.markdown("""
<div style='
    background: linear-gradient(135deg, #2563eb, #7c3aed);
    padding: 25px;
    border-radius: 16px;
    text-align: center;
    margin-bottom: 25px;
    box-shadow: 0 0 25px rgba(124,58,237,0.4);
'>
    <h1 style='color:white;margin:0;'>⚡ TextMorph</h1>
    <p style='color:#e0e7ff;margin:0;'>AI-Powered Text Intelligence Suite</p>
</div>
""", unsafe_allow_html=True)

   # Create centered layout
   col1, col2, col3 = st.columns([1, 2, 1])

   with col2:
    st.markdown("""
        <div style="background: rgba(255,255,255,0.05); padding: 30px; border-radius: 20px; border: 1px solid rgba(255,255,255,0.1);">
            <h2 style='text-align: center; color: white; margin-bottom: 20px;'>🔐 Secure Login</h2>
        """, unsafe_allow_html=True)

    with st.form("login_form"):
      email = st.text_input("Email *")
      password = st.text_input("Password *", type='password')
      submit = st.form_submit_button("Login", use_container_width=True)

      if submit:
        is_locked, wait_time = db.is_rate_limited(email)
        if is_locked:
          st.error(f"⛔ Locked! Try again in {int(wait_time)}s.")
        elif db.authenticate_user(email, password):
          st.session_state['user'] = email
          set_user_active(email)
          log_system_event("LOGIN", f"{email} logged in")
          st.rerun()
        else:
          st.error("Invalid email or password.")

    # Buttons INSIDE centered column
    c1, c2 = st.columns(2)

    with c1:
        if st.button("Create Account", use_container_width=True):
            switch_page("register")

    with c2:
        if st.button("Forgot Password", use_container_width=True):
            switch_page("forgot_password")

def forgot_password_page():

    # Centered Title
    st.markdown("<h1 style='text-align: center;'>🔑 Reset Password</h1>", unsafe_allow_html=True)

    # Center layout
    col1, col2, col3 = st.columns([1, 2, 1])

    with col2:

        if "otp_sent" not in st.session_state:
            st.session_state.otp_sent = False

        email = st.text_input("Enter your registered email")

        # Step 1: Send OTP
        if not st.session_state.otp_sent:

            if st.button("Send OTP", use_container_width=True):

                if not is_valid_email(email):
                    st.error("Enter valid email")
                    return

                otp = generate_otp()

                st.session_state.reset_email = email
                st.session_state.reset_otp = otp
                st.session_state.otp_expiry = datetime.datetime.now() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)

                if send_otp(email, otp):
                    st.session_state.otp_sent = True
                    st.success("OTP sent to your email")

        # Step 2: Verify OTP
        else:

            otp_input = st.text_input("Enter OTP")
            new_password = st.text_input("New Password", type="password")

            if st.button("Reset Password", use_container_width=True):

                if datetime.datetime.now() > st.session_state.otp_expiry:
                    st.error("OTP expired")
                    st.session_state.otp_sent = False
                    return

                if otp_input == st.session_state.reset_otp:

                    db.update_password(st.session_state.reset_email, new_password)

                    st.success("Password updated successfully")

                    st.session_state.otp_sent = False

                    switch_page("login")

                else:
                    st.error("Invalid OTP")

        # Back button (centered nicely)
        if st.button("⬅ Back to Login", use_container_width=True):
            switch_page("login")
def generate_otp():
    return str(random.randint(100000, 999999))
def send_otp(email, otp):

    subject = "TextMorph Password Reset OTP"

    body = f"""
Your OTP for resetting your password is: {otp}

This OTP will expire in {OTP_EXPIRY_MINUTES} minutes.

If you did not request this, ignore this email.
"""

    msg = MIMEMultipart()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = email
    msg["Subject"] = subject

    msg.attach(MIMEText(body, "plain"))

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.send_message(msg)
        server.quit()
        return True
    except Exception as e:
        st.error(f"Email error: {e}")
        return False
def update_password(email, new_password):

    hashed = bcrypt.hashpw(new_password.encode(), bcrypt.gensalt())

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("UPDATE users SET password=? WHERE email=?", (hashed, email))

    conn.commit()
    conn.close()

def register_page():
    # Centered Title
    st.markdown("<h1 style='text-align: center;'>📝 Create Account</h1>", unsafe_allow_html=True)

    # Center layout
    col1, col2, col3 = st.columns([1, 2, 1])

    with col2:

        with st.form("register_form"):

            email = st.text_input("Email Address *")
            password = st.text_input("Password *", type='password')
            confirm_password = st.text_input("Confirm Password *", type='password')

            submit = st.form_submit_button("Create Account", use_container_width=True)

            if submit:

                # Email validation
                if not is_valid_email(email):
                    st.error("Enter a valid email")
                    return

                # Password match
                if password != confirm_password:
                    st.error("Passwords do not match")
                    return

                # Password strength
                strength, suggestions = check_password_strength(password)
                if strength == "Weak":
                    st.error("Weak password. " + ", ".join(suggestions))
                    return

                # Register user
                if db.register_user(email, password):
                    st.success("✅ Account created successfully")
                    switch_page("login")
                else:
                    st.error("User already exists")

        # Back button
        if st.button("⬅ Back to Login", use_container_width=True):
            switch_page("login")

if st.session_state['user']:
  with st.sidebar:
    st.image(
            "https://upload.wikimedia.org/wikipedia/commons/thumb/9/95/Infosys_logo.svg/1280px-Infosys_logo.svg.png",
            width=150
        )
    email = st.session_state['user']
    user = db.get_user_profile(email)
    avatar = user[1] if user else None
    if avatar and os.path.exists(avatar):
      st.image(avatar, width=80)
    else:
      st.image("https://cdn-icons-png.flaticon.com/512/149/149071.png",
                width=80
            )
    st.markdown(f"**{email}**")
    st.markdown("---")
    if st.session_state['user'] == "admin@llm.com":
      opts = ["Profile","Admin"]
      icons = ["person","shield-lock"]
    else:
      opts = ["Summarize", "Paraphrase", "Readability", "Tune", "History", "Profile"]
      icons = ["file-text", "arrow-repeat", "book", "sliders", "clock-history", "person"]
    selected = option_menu(
            "Text Morph",
            opts,
            icons=icons,
            menu_icon="cast",
            default_index=0,
            styles={
              "container": {
                  "background-color": "#020617",
                  "padding": "10px"
              },
              "icon": {
                  "color": "#3b82f6",
                  "font-size": "18px"
              },
              "nav-link": {
                  "color": "#94a3b8",
                  "font-size": "14px",
                  "margin": "5px 0px",
                  "border-radius": "8px",
                  "padding": "8px 12px",
              },
              "nav-link-selected": {
                  "background": "linear-gradient(135deg, #2563eb, #1d4ed8)",
                  "color": "white",
                  "border-radius": "8px",
                "font-weight": "600",
              },
            }
      )
    st.markdown("---")
    if st.button("🔓 Log Out"):
      logout()

    # Clear stale results
  _clear_stale_results(selected)
  if selected == "Profile":
    profile_page()
  elif selected == "Summarize" and email != "admin@llm.com":
    summarizer_page()

  elif selected == "Paraphrase" and email != "admin@llm.com":
    paraphraser_page()

  elif selected == "Readability" and email != "admin@llm.com":
    readability_page()

  elif selected == "Tune" and email != "admin@llm.com":
    augmentation_page()

  elif selected == "History" and email != "admin@llm.com":
    history_page()

  elif selected == "Admin":
    admin_page()

else:
  if st.session_state['page'] == "login":
    login_page()

  elif st.session_state['page'] == "register":
    register_page()

  elif st.session_state['page'] == "forgot_password":
    forgot_password_page()

Overwriting app.py


In [ ]:
import os
import subprocess
import time
from google.colab import userdata
from google.colab import drive
from pyngrok import ngrok
from datasets import load_dataset
from google.colab._message import MessageError

# 0. Mount drive early to ensure datasets save successfully
try:
    drive.mount("/content/drive", force_remount=True, timeout_ms=120000)
except ValueError:
    print("Drive already mounted or mountpoint error ignored.")
except MessageError as e:
    print(f"⚠️ Google Drive mount failed: {e}. Please ensure you authorize access when prompted and try re-running the cell.")
except Exception as e:
    print(f"⚠️ An unexpected error occurred during Google Drive mount: {e}")

# 1. Download Datasets if they do not exist
save_dir = "/content/drive/MyDrive/TextMorph/Datasets"
os.makedirs(save_dir, exist_ok=True)

print("📥 Ensuring Datasets are downloaded to Google Drive...")
if not os.path.exists(f"{save_dir}/cnn_dailymail"):
    print("Downloading CNN/DailyMail...")
    try:
        dataset_cnn = load_dataset("cnn_dailymail", "3.0.0")
        dataset_cnn.save_to_disk(f"{save_dir}/cnn_dailymail")
    except: print("Warning: Could not download CNN")
else:
    print("✅ CNN/DailyMail dataset already exists. Skipping download.")

if not os.path.exists(f"{save_dir}/xsum"):
    print("Downloading XSum...")
    try:
        dataset_xsum = load_dataset("xsum")
        dataset_xsum.save_to_disk(f"{save_dir}/xsum")
    except: print("Warning: Could not download XSum")
else:
    print("✅ XSum dataset already exists. Skipping download.")

if not os.path.exists(f"{save_dir}/paws"):
    print("Downloading PAWS...")
    try:
        dataset_paws = load_dataset("paws", "labeled_final")
        dataset_paws.save_to_disk(f"{save_dir}/paws")
    except: print("Warning: Could not download PAWS")
else:
    print("✅ PAWS dataset already exists. Skipping download.")

print(f"🎉 All datasets ready in {save_dir}")

# 2. Retrieve secrets safely
email_pass = None
ngrok_token = "3AbxG0gCiTPdX0Z6B03TdP6V5kq_eCLGFZwSiqBiWfFz9YmD"  # 👈 ADD HERE

os.environ["JWT_SECRET"] = "super-secret-change-me"

# 3. Authenticate Ngrok
if ngrok_token:
    # Ensure any previous ngrok tunnels are closed and processes killed
    try:
        ngrok.disconnect()
    except:
        pass # Ignore if no tunnel is active
    ngrok.kill()
    time.sleep(2) # Give a bit more time for processes to terminate

    ngrok.set_auth_token(ngrok_token)

    # 4. Run Streamlit
    process = subprocess.Popen(["streamlit", "run", "app.py"], env=os.environ.copy())
    time.sleep(5) # Give Streamlit more time to start

    # 5. Open Tunnel
    try:
        # Check if streamlit process is still running
        if process.poll() is not None:
            print("❌ Streamlit app terminated unexpectedly before ngrok tunnel could be established.")
            raise Exception("Streamlit app failed to start.")

        public_url = ngrok.connect(8501).public_url
        print(f"🚀 App Running: {public_url}")
        print("👇 Click the link above!")
    except Exception as e:
        print(f"❌ Ngrok Error: {e}")
        # Try to clean up the streamlit process if ngrok connection failed
        if process.poll() is None:
            process.terminate()
            print("Streamlit process terminated due to ngrok error.")

    # 6. Keep Alive
    try: input("🛑 Press ENTER to STOP...")
    except: pass
    finally:
        if process.poll() is None:
            process.terminate()
        ngrok.kill()
        print("✅ Stopped.")
else:
    print("❌ No Ngrok Token found. Add 'NGROK_AUTHTOKEN' to Secrets.")

⚠️ Google Drive mount failed: Error: credential propagation was unsuccessful. Please ensure you authorize access when prompted and try re-running the cell.
📥 Ensuring Datasets are downloaded to Google Drive...
✅ CNN/DailyMail dataset already exists. Skipping download.
✅ XSum dataset already exists. Skipping download.
✅ PAWS dataset already exists. Skipping download.
🎉 All datasets ready in /content/drive/MyDrive/TextMorph/Datasets
🚀 App Running: https://angelita-managerial-westin.ngrok-free.dev
👇 Click the link above!
